In [4]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("resume.pdf")
docs=loader.load()
docs


/var/folders/5r/mxfdbf617wn355ck4vvwbpp00000gn/T/ipykernel_87253/534207984.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)


[Document(metadata={'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260616220846Z00'00'", 'moddate': "D:20260616220846Z00'00'", 'source': 'resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='RAJESH NALLA 361-504-6181 | rajeshna.dev01@gmail.com | https://www.linkedin.com/in/rajndev  | Carlisle, PA PROFESSIONAL SUMMARY • Software Engineer with 5+ years of experience building internal platforms, workflow systems, and production web applications using TypeScript, Node.js, Python, PostgreSQL, and AWS. • Delivered systems processing 2M+ daily transactions, improved latency from 220 milliseconds to 95 milliseconds, and raised automated test coverage from 42% to 88% across 12+ services. • Strong background in REST APIs, relational data modeling, authorization controls, audit-friendly workflows, third-party integrations, and scalable operational software for regulated environments. TECHNICAL SKILLS Programming:

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitters=RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
text_splitters.split_documents(docs)

[Document(metadata={'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260616220846Z00'00'", 'moddate': "D:20260616220846Z00'00'", 'source': 'resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='RAJESH NALLA 361-504-6181 | rajeshna.dev01@gmail.com | https://www.linkedin.com/in/rajndev  |'),
 Document(metadata={'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260616220846Z00'00'", 'moddate': "D:20260616220846Z00'00'", 'source': 'resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='| Carlisle, PA PROFESSIONAL SUMMARY • Software Engineer with 5+ years of experience building'),
 Document(metadata={'producer': 'macOS Version 26.5.1 (Build 25F80) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20260616220846Z00'00'", 'moddate': "D:20260616220846Z00'00'", 'source': 'resume.pdf', 'total_pages': 2, 'page': 0, 'page_label':

In [6]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

db=FAISS.from_documents(docs,OpenAIEmbeddings())
db

/var/folders/5r/mxfdbf617wn355ck4vvwbpp00000gn/T/ipykernel_87253/3972146939.py:4: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  db=FAISS.from_documents(docs,OpenAIEmbeddings())


In [7]:
query="Health care"
result=db.similarity_search(query)
result[0].page_content

'• Resolved recurring platform failures and service regressions, lowering issue recurrence by 26% across cloud-hosted business applications with strict operational reliability requirements. • Collaborated with product, operations, and engineering stakeholders to translate ambiguous requirements into scalable technical solutions across deadline-driven business initiatives. Accenture — Associate Soeware Engineer       Jul 2020 to Nov 2022 • Modernized Java and Python backend services for four enterprise clients, delivering releases 20% ahead of schedule across large-scale platform modernization programs. • Streamlined CI/CD pipelines and deployment validation workflows, shortening release cycles from 30 minutes to 12 minutes across shared engineering environments and delivery processes. • Configured event-driven integrations and service interfaces supporting $2M+ in daily transaction volume per client at sub-200 millisecond processing latency. • Investigated processing bottlenecks and in

In [8]:
from langchain_ollama import OllamaLLM
## load ollama llm model
llm=OllamaLLM(model="llama2")
llm

OllamaLLM(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, model='llama2')

In [9]:
## Desing the chat template
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
Answer the following question based only on the provided content. 
Think step by step before providing a detailed answer.
I will tip you $100 if the user finds the answer helpful. 
<context>
{context}
</context>
Question: {input}""")

In [10]:
## Chain INtroduction
### Create Stuff Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm,prompt)

In [11]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided content. \nThink step by step before providing a detailed answer.\nI will tip you $100 if the user finds the answer helpful. \n<context>\n{context}\n</context>\nQuestion: {input}'), additional_kwargs={})])
| OllamaLLM(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, model='llama2')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [12]:
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x124154ad0>, search_kwargs={})

In [17]:
## retriver chain
from langchain_classic.chains import create_retrieval_chain
retriever_chain=create_retrieval_chain(retriever, document_chain)


In [19]:
response=retriever_chain.invoke({"input":"university"})


In [20]:
response['answer']

'Based on the provided resume, the individual\'s university name is:\n\nUniversity of Central Missouri\n\nThe name of the university is mentioned in the resume under the "Education" section, where it states that the individual earned a Master of Science degree in Computer Science from the University of Central Missouri in May 2024.'